## 1. Data Acquisition and Preprocessing

### Objective

Construct two datasets:

- **Genome-wide dataset (chr20–22)** for GRM construction and PCA.
- **chr22-only dataset** for association testing.

In [1]:
!plink --version

PLINK v1.9.0-b.8 64-bit (22 Oct 2024)


In [6]:
%%bash
# 1.1 Convert VCF to PLINK format

set -euo pipefail

mkdir -p data_preprocessed

# 1) Convert each chromosome VCF (chr20–22) to PLINK bed/bim/fam
for chr in {20..22}; do
  echo "Converting chr${chr} VCF to PLINK..."
  plink --vcf data/ALL.chr${chr}.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz \
        --make-bed \
        --biallelic-only strict list \
        --snps-only just-acgt \
        --out data_preprocessed/tmp.chr${chr}
done

# Rename variant IDs in each .bim to ensure uniqueness (CHR:BP:A1:A2)
for chr in 20 21 22; do
  awk '{ $2 = $1 ":" $4 ":" $5 ":" $6; print }' data_preprocessed/tmp.chr${chr}.bim > data_preprocessed/tmp.chr${chr}.bim.tmp
  mv data_preprocessed/tmp.chr${chr}.bim.tmp data_preprocessed/tmp.chr${chr}.bim
done

# 2) Build genome-wide dataset (chr20–22)
#    Use chr20 as base and merge chr21–22.
merge_list=data_preprocessed/merge_list.txt
: > "${merge_list}"
for chr in {21..22}; do
  echo "data_preprocessed/tmp.chr${chr}.bed data_preprocessed/tmp.chr${chr}.bim data_preprocessed/tmp.chr${chr}.fam" >> "${merge_list}"
done

plink --bfile data_preprocessed/tmp.chr20 \
      --merge-list "${merge_list}" \
      --make-bed \
      --out data_preprocessed/genome_raw

# 3) Extract chr22-only dataset from genome-wide set
plink --bfile data_preprocessed/genome_raw \
      --chr 22 \
      --make-bed \
      --out data_preprocessed/chr22_raw


Converting chr20 VCF to PLINK...


PLINK v1.9.0-b.8 64-bit (22 Oct 2024)              cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data_preprocessed/tmp.chr20.log.
Options in effect:
  --biallelic-only strict list
  --make-bed
  --out data_preprocessed/tmp.chr20
  --snps-only just-acgt
  --vcf data/ALL.chr20.phase3_shapeit2_mvncall_integrated_v5a.20130502.genotypes.vcf.gz

16384 MB RAM detected; reserving 8192 MB for main workspace.
--vcf: data_preprocessed/tmp.chr20-temporary.bed +
data_preprocessed/tmp.chr20-temporary.bim +
data_preprocessed/tmp.chr20-temporary.fam written.
(8972 variants skipped.)
1739315 out of 1803869 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to data_preprocessed/tmp.chr20.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121

position.
position.
position.


Performing single-pass merge (2504 people, 3849216 variants).
Merged fileset written to data_preprocessed/genome_raw-merge.bed +
data_preprocessed/genome_raw-merge.bim + data_preprocessed/genome_raw-merge.fam
.
3849216 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to data_preprocessed/genome_raw.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is exactly 1.
3849216 variants and 2504 people pass filters and QC.
Note: No phenotypes present.
--make-bed to data_preprocessed/genome_raw.bed +
data_preprocessed/genome_raw.bim + data_preprocessed/genome_raw.fam ... 101112131415161718192021222324

In [ ]:
# 1.2 Quality Control

In [7]:
%%bash
# 1.2 Quality Control

set -euo pipefail

mkdir -p data_preprocessed

# Genome-wide QC on merged chr20–22 dataset
plink --bfile data_preprocessed/genome_raw \
      --maf 0.05 \
      --geno 0.02 \
      --hwe 1e-6 \
      --make-bed \
      --out data_preprocessed/genome_qc

# QC on chr22-only dataset
plink --bfile data_preprocessed/chr22_raw \
      --maf 0.05 \
      --geno 0.02 \
      --make-bed \
      --out data_preprocessed/chr22_qc


PLINK v1.9.0-b.8 64-bit (22 Oct 2024)              cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data_preprocessed/genome_qc.log.
Options in effect:
  --bfile data_preprocessed/genome_raw
  --geno 0.02
  --hwe 1e-6
  --maf 0.05
  --make-bed
  --out data_preprocessed/genome_qc

16384 MB RAM detected; reserving 8192 MB for main workspace.
3849216 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to data_preprocessed/genome_qc.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is exactly 1.
0 variants removed due to missing g

In [8]:
%%bash
# 1.3 LD Pruning for PCA and GRM Stability

set -euo pipefail

# Identify LD-independent SNPs (window 50, step 5, r² 0.2)
plink --bfile data_preprocessed/genome_qc \
      --indep-pairwise 50 5 0.2 \
      --out data_preprocessed/genome_pruned

# Extract pruned SNPs into new dataset
plink --bfile data_preprocessed/genome_qc \
      --extract data_preprocessed/genome_pruned.prune.in \
      --make-bed \
      --out data_preprocessed/genome_pruned_data

PLINK v1.9.0-b.8 64-bit (22 Oct 2024)              cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data_preprocessed/genome_pruned.log.
Options in effect:
  --bfile data_preprocessed/genome_qc
  --indep-pairwise 50 5 0.2
  --out data_preprocessed/genome_pruned

16384 MB RAM detected; reserving 8192 MB for main workspace.
223073 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to data_preprocessed/genome_pruned.nosex .
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is exactly 1.
223073 variants and 2504 people pass filters and Q

## 2. Population Structure Covariates (PCA)

In [9]:
# 2.1 Run PCA with PLINK (top 10 PCs)
!plink --bfile data_preprocessed/genome_pruned_data \
       --pca 10 \
       --out data_preprocessed/pca_results

# 2.2 Format PCA output as covariate file
# pca_results.eigenvec has columns: FID IID PC1 PC2 ...

eigenvec_path = "data_preprocessed/pca_results.eigenvec"
covariate_path = "data_preprocessed/covariates.txt"

with open(eigenvec_path, "r") as fin, open(covariate_path, "w") as fout:
    # Write header with first 5 PCs
    fout.write("FID IID PC1 PC2 PC3 PC4 PC5\n")
    for line in fin:
        parts = line.strip().split()
        if len(parts) < 7:
            continue  # skip malformed lines
        fid, iid = parts[0], parts[1]
        pcs = parts[2:7]  # PC1–PC5
        fout.write(f"{fid} {iid} {' '.join(pcs)}\n")

print(f"Wrote covariates to {covariate_path}")

PLINK v1.9.0-b.8 64-bit (22 Oct 2024)              cog-genomics.org/plink/1.9/
(C) 2005-2024 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to data_preprocessed/pca_results.log.
Options in effect:
  --bfile data_preprocessed/genome_pruned_data
  --out data_preprocessed/pca_results
  --pca 10

16384 MB RAM detected; reserving 8192 MB for main workspace.
21032 variants loaded from .bim file.
2504 people (0 males, 0 females, 2504 ambiguous) loaded from .fam.
Ambiguous sex IDs written to data_preprocessed/pca_results.nosex .
Using up to 8 threads (change this with --threads).
Before main variant filters, 2504 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is exactly 1.
21032 variants and 2504 people pass filters and QC.
Note: No phenotyp